In [1]:
import numpy as np
import librosa, librosa.display, music21
from music21 import note, stream

/Users/amishasingh/Documents/sheetgenie/nsynth/.venv/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [30]:
import os
files = [
    "keyboard_acoustic_000-050-127.wav",
    "keyboard_acoustic_003-044-127.wav",
    "keyboard_acoustic_007-048-075.wav",
]
def create_audio_file(files):
    combined_audio=[]
    for f in files:
        path = os.path.join("nsynth-train/audio",f)
        y,sr = librosa.load(path, sr=16000)

        '''
        adding silences to make note changes more distinct
        '''
        silence=np.zeros(int(0.2*sr)) #200ms gap

        combined_audio.append(y)
        combined_audio.append(silence)
    return np.concatenate(combined_audio)

y_combined = create_audio_file(files)




In [6]:
import soundfile as sf
sf.write("./seq/seq3.wav",y_combined,sr=16000)

In [7]:
import joblib
model = joblib.load("rf_13_classifier.pkl")

In [8]:
def predict_frames(y_audio, sr):
    frame_length=int(0.05*sr) #50ms frames
    hop_length=frame_length

    pitches=[]
    for i in range(0,len(y_audio), hop_length):
        frame=y_audio[i:i+frame_length]
        if len(frame)<frame_length:
            continue

        feature = np.hstack((
            np.mean(librosa.feature.mfcc(y=frame, sr=sr, n_mfcc=13).T, axis=0),
            np.mean(librosa.feature.chroma_stft(y=frame, sr=sr).T, axis=0),
            np.mean(librosa.feature.spectral_centroid(y=frame, sr=sr)),
            np.mean(librosa.feature.zero_crossing_rate(frame))
        )).reshape(1,-1)

        pitch=model.predict(feature)[0]
        pitches.append(pitch)

    return pitches

In [9]:
def segment_audio_onsets(y, sr):

    # detect note boundaries
    onset_frames = librosa.onset.onset_detect(
    y=y,
    sr=sr,
)

    onset_samples = librosa.frames_to_samples(onset_frames)

    segments = []

    for i in range(len(onset_samples)-1):
        start = onset_samples[i]
        end = onset_samples[i+1]
        segments.append((start, end))

    # last segment
    segments.append((onset_samples[-1], len(y)))

    return segments

In [10]:
def predict_segments(y, sr, segments):

    notes = []
    durations = []

    for start, end in segments:

        segment = y[start:end]

        # skip silence
        if np.mean(np.abs(segment)) < 0.01:
            continue

        if len(segment) < 1000:
            continue

        feature = np.hstack((
            np.mean(librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13).T, axis=0),
            np.mean(librosa.feature.chroma_stft(y=segment, sr=sr).T, axis=0),
            np.mean(librosa.feature.spectral_centroid(y=segment, sr=sr)),
            np.mean(librosa.feature.zero_crossing_rate(segment))
        )).reshape(1, -1)

        pitch = model.predict(feature)[0]

        duration = (end - start) / sr

        notes.append(pitch)
        durations.append(duration)

    return notes, durations

In [11]:
def seconds_to_beats(d, bpm=120):
    return d / (60 / bpm)

In [12]:
def quantize_duration(d):
    vals=[0.25,0.5,1,2,4,8]
    return min(vals, key=lambda x: abs(x-d))

def create_seq_sheet(notes, durations,filename):

    s = stream.Stream()

    for p, d in zip(notes, durations):

        n = note.Note()
        n.pitch.midi = int(p)
        beats=seconds_to_beats(d,bpm=120)
        n.quarterLength=quantize_duration(beats)

        s.append(n)

    s.write("musicxml", "seq/"+filename+".xml")

    print("Sheet music created at seq/"+filename+".xml")

In [19]:
y, sr = librosa.load("seq/seq3.wav", sr=16000)

segments = segment_audio_onsets(y, sr)

notes, durations = predict_segments(y, sr, segments)

create_seq_sheet(notes, durations,"sheet3")

Sheet music created at seq/sheet3.xml


In [20]:
print(sum(durations), len(y)/sr)

12.440000000000001 12.6


In [21]:
import pandas as pd

df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/notes_durations3.csv", index=True)

In [34]:
## **TESTS**

In [27]:
''' 
test 1 -> 3 notes with 2 repeated
'''
files = [
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_003-048-127.wav",
]

y_combined = create_audio_file(files)
sf.write("./seq/test1.wav",y_combined,16000)
y, sr = librosa.load("seq/test1.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
create_seq_sheet(notes, durations,"test1_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test_notes1.csv", index=True)


Sheet music created at seq/test1_sheet.xml
11.296 12.6


In [28]:
''' 
test 2 -> 5 notes
'''

files = [
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_003-048-127.wav",
    "keyboard_acoustic_007-050-075.wav",
    "keyboard_acoustic_011-053-127.wav",
    "keyboard_acoustic_018-045-127.wav"
]

y_combined = create_audio_file(files)
sf.write("./seq/test2.wav",y_combined,16000)
y, sr = librosa.load("seq/test2.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
create_seq_sheet(notes, durations,"test2_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test2_notes.csv", index=True)


Sheet music created at seq/test2_sheet.xml
12.488 21.0


In [31]:
'''
test 3 -> stretching one note
'''
files = [
    "keyboard_acoustic_000-050-127.wav",
    "keyboard_acoustic_003-044-127.wav",
    "keyboard_acoustic_007-048-075.wav",
]
def create_test3_file(files):
    combined_audio=[]
    for f in files:
        path = os.path.join("nsynth-train/audio",f)
        y,sr = librosa.load(path, sr=16000)
        if files.index(f)==0:
            y = librosa.effects.time_stretch(y, rate=0.5)  # longer note
        
        '''
        adding silences to make note changes more distinct
        '''
        silence=np.zeros(int(0.2*sr)) #200ms gap

        combined_audio.append(y)
        combined_audio.append(silence)
    return np.concatenate(combined_audio)

y_combined = create_audio_file(files)
sf.write("./seq/test3.wav",y_combined,16000)
y, sr = librosa.load("seq/test3.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
create_seq_sheet(notes, durations,"test3_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test3_notes.csv", index=True)


Sheet music created at seq/test3_sheet.xml
12.440000000000001 12.6


In [33]:
'''
test 4 -> removing silences between notes
'''
files = [
    "keyboard_acoustic_000-050-127.wav",
    "keyboard_acoustic_003-044-127.wav",
    "keyboard_acoustic_007-048-075.wav",
]
def create_test4_file(files):
    combined_audio=[]
    for f in files:
        path = os.path.join("nsynth-train/audio",f)
        y,sr = librosa.load(path, sr=16000)
        combined_audio.append(y)

    return np.concatenate(combined_audio)

y_combined = create_test4_file(files)
sf.write("./seq/test4.wav",y_combined,16000)
y, sr = librosa.load("seq/test4.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
create_seq_sheet(notes, durations,"test4_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test4_notes.csv", index=True)


Sheet music created at seq/test4_sheet.xml
11.84 12.0
